In [4]:
import os
import pyspark

# Auto-detect Spark version → correct connector
spark_version = pyspark.__version__
print(f"Detected PySpark version: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
else:
    # Spark 3.x
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

print(f"Using connector:         {KAFKA_PACKAGE}")

# Must be set BEFORE SparkSession.builder
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {KAFKA_PACKAGE} pyspark-shell'

Detected PySpark version: 4.0.0.dev2
Using connector:         org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2


In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

Spark 4.0.0-preview2 — ready


In [8]:
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load()
)

kafka_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [9]:
from pyspark.sql.functions import col

batch_counter = {"n": 0}

def peek_raw(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id}: {df.count()} messages ---")
    df.select(
        "topic",
        "partition",
        "offset",
        "timestamp",
        col("key").cast("string").alias("key"),
        col("value").cast("string").alias("value"),   # bytes → UTF-8 text
    ).show(5, truncate=100)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = (
    kafka_raw.writeStream
    .foreachBatch(peek_raw)
    .option("checkpointLocation", "/tmp/chk_lab4_peek")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

--- Batch 0: 942 messages ---
+------------+---------+------+-----------------------+----+----------------------------------------------------------------------------------------------------+
|       topic|partition|offset|              timestamp| key|                                                                                               value|
+------------+---------+------+-----------------------+----+----------------------------------------------------------------------------------------------------+
|transactions|        0|     0|2026-06-02 19:27:46.625|NULL|{"tx_id": "TX2118", "user_id": "u06", "amount": 3470.48, "store": "Warszawa", "category": "\u017c...|
|transactions|        0|     1|2026-06-02 19:27:47.625|NULL|{"tx_id": "TX3810", "user_id": "u09", "amount": 4628.87, "store": "Gda\u0144sk", "category": "\u0...|
|transactions|        0|     2|2026-06-02 19:27:48.626|NULL|{"tx_id": "TX8819", "user_id": "u05", "amount": 369.63, "store": "Wroc\u0142aw", "category": "ksi...

In [10]:
# Step 1: binary → string (raw JSON as text)
step1 = kafka_raw.select(
    col("offset"),
    col("partition"),
    col("value").cast("string").alias("raw_json"),
)

batch_counter["n"] = 0

def show_step1(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(3, truncate=120)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = step1.writeStream.foreachBatch(show_step1) \
         .option("checkpointLocation", "/tmp/chk_lab4_step1").start()
try:
    q.awaitTermination()
except:
    q.stop()

In [11]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

# Step 2: string → struct (single 'tx' column containing all fields)
step2 = kafka_raw.select(
    from_json(col("value").cast("string"), tx_schema).alias("tx")
)

batch_counter["n"] = 0

def show_step2(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} — schema after from_json ---")
    df.printSchema()
    df.show(3, truncate=False)
    if batch_counter["n"] >= 1:
        raise Exception("stop")

q = step2.writeStream.foreachBatch(show_step2) \
         .option("checkpointLocation", "/tmp/chk_lab4_step2").start()
try:
    q.awaitTermination()
except:
    q.stop()

In [12]:
from pyspark.sql.functions import to_timestamp

# Step 3: struct → flat columns + timestamp conversion
df = (
    kafka_raw
    .select(from_json(col("value").cast("string"), tx_schema).alias("tx"))
    .select("tx.*")
    .withColumn("timestamp", to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"))
)

print("Final schema:")
df.printSchema()

batch_counter["n"] = 0

def show_parsed(df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(5, truncate=False)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = df.writeStream.foreachBatch(show_parsed) \
      .option("checkpointLocation", "/tmp/chk_lab4_parsed").start()
try:
    q.awaitTermination()
except:
    q.stop()

Final schema:
root
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- store: string (nullable = true)
 |-- category: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [13]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

windowed = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "1 minute"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount"),
    )
)

batch_counter["n"] = 0

def show_window(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    (
        df.select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "store", "tx_count", "total_amount",
        )
        .orderBy("from", "store")
        .show(truncate=False)
    )
    if batch_counter["n"] >= 5:
        raise Exception("stop")

q = (
    windowed.writeStream
    .outputMode("append")
    .foreachBatch(show_window)
    .option("checkpointLocation", "/tmp/chk_lab4_windows")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

In [14]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round, col

windowed = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "1 minute"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount"),
    )
)

batch_counter["n"] = 0

def show_window(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    (
        df.select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "store", "tx_count", "total_amount",
        )
        .orderBy("from", "store")
        .show(truncate=False)
    )
    if batch_counter["n"] >= 5:
        raise Exception("stop")

In [15]:
from pyspark.sql.functions import to_json, struct, lit

alerts = (
    df
    .filter(col("amount") > 3000)
    .select(
        to_json(
            struct(
                "tx_id", "user_id", "amount", "store", "category",
                col("timestamp").cast("string"),
                lit("HIGH").alias("alert_level"),
            )
        ).alias("value")    # Kafka requires a 'value' column
    )
)

alert_query = (
    alerts.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")
    .option("checkpointLocation", "/tmp/chk_lab4_alerts")
    .outputMode("append")
    .start()
)
print("Alert stream started. Stop manually: alert_query.stop()")

Alert stream started. Stop manually: alert_query.stop()


## Homework

### HW1

In [16]:
# Homework 1: Sliding window (2 minutes / 1-minute step) on the Kafka stream per store.

sliding_store = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "2 minutes", "1 minute"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount"),
    )
)

batch_counter["n"] = 0

def show_sliding(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} — sliding 2min / 1min step ===")
    (
        df.select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "store", "tx_count", "total_amount",
        )
        .orderBy("from", "store")
        .show(truncate=False)
    )
    if batch_counter["n"] >= 4:
        raise Exception("stop")

q = (
    sliding_store.writeStream
    .outputMode("append")
    .foreachBatch(show_sliding)
    .option("checkpointLocation", "/tmp/chk_lab4_hw1")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

### HW2

In [18]:
# Homework 2: Add a 'ratio' field = amount / 400.0 (approximate average from Lab 2) to alerts.

from pyspark.sql.functions import to_json, struct, lit

alerts_with_ratio = (
    df
    .filter(col("amount") > 3000)
    .withColumn("ratio", _round(col("amount") / 400.0, 2))
    .select(
        to_json(
            struct(
                "tx_id", "user_id", "amount", "store", "category",
                col("timestamp").cast("string"),
                "ratio",                        # ← field mới
                lit("HIGH").alias("alert_level"),
            )
        ).alias("value")
    )
)

batch_counter["n"] = 0

def show_alerts(df, batch_id):
    batch_counter["n"] += 1
    print(f"\n--- Batch {batch_id} — alerts JSON preview ---")
    df.show(5, truncate=False)
    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = (
    alerts_with_ratio.writeStream
    .foreachBatch(show_alerts)
    .option("checkpointLocation", "/tmp/chk_lab4_hw2_preview")
    .start()
)
try:
    q.awaitTermination()
except:
    q.stop()

# Bây giờ ghi vào Kafka topic 'alerts'
alert_query2 = (
    alerts_with_ratio.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")
    .option("checkpointLocation", "/tmp/chk_lab4_hw2_kafka")
    .outputMode("append")
    .start()
)

import time
time.sleep(20)
alert_query2.stop()
print("HW2 alert stream (with ratio) finished.")

HW2 alert stream (with ratio) finished.


### HW3: What happens to the results when you stop the producer and wait 2 minutes? Why?

**Answer**

When the producer is stopped, no new transactions are written to the `transactions` topic. The streaming query continues to run, but it no longer receives fresh events.

In the first tens of seconds after stopping the producer, Spark may still emit some results for windows that were already “in progress”, because the watermark can advance based on the event-time of the last messages that had already arrived. Once those are processed, event-time stops progressing and the watermark effectively freezes at the last observed event-time.

After about two minutes with no new data:

- **In `append` output mode**, windows never close, because the watermark does not advance past their end time. As a result, no new rows are ever appended to the sink. The query keeps running but produces no further output.
- **In `complete` output mode**, Spark recomputes and outputs the full aggregated table every micro‑batch, but since there is no new input, the contents of that table do not change. It looks like the results are “stuck” on the last state.

Spark itself does not crash or stop; it simply waits for new data. As soon as the producer starts sending events again, event‑time begins to progress, the watermark moves forward, windows can be closed, and new results appear. Because the query uses a `checkpointLocation`, it resumes correctly from the previous state instead of starting from scratch.

The underlying reason is that Spark Structured Streaming drives watermarks and window completion by **event time** (the `timestamp` column in the data), not by wall‑clock time. If no new events arrive, event time does not advance, the watermark does not move, and therefore windows never satisfy the condition “watermark > window end”. This behavior is intentional so that late‑arriving data can still be handled correctly, regardless of when it physically arrives.